In [13]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from GenerateTrainingData import GenerateTrainingData

In [14]:
# divide the nodes up so they get an equal number of features: n
# train a common ML model that takes in n features and makes a prediction
# from our understanding of the underlying distribution, each node should be learning a different function
# so in theory this should have terrible performance

# 1. Initialize random weights for ridge regression model (global)
# 2. Send weights out 
# 3. Run ridge regression on the data points with this initial model
# 4. Send the weights back to the PS. Average the weights
# 5. Repeat steps 2 - 4 for a number of iterations or convergence criterion

In [15]:
NUM_NODES = 10
ALPHA = 1.0 
STD_DEV = 0.31622776601 # sqrt 0.1
NUM_FEATURES = 120
NUM_TRAIN_DATA_POINTS = 1000
NUM_TEST_DATA_POINTS = 10000   

data_gen = GenerateTrainingData(num_features=NUM_FEATURES, STD_DEV=STD_DEV)
A_train, B_train, C_train = data_gen.generate_data(num_data_points=NUM_TRAIN_DATA_POINTS)
A_test, B_test, C_test = data_gen.generate_data(num_data_points=NUM_TEST_DATA_POINTS)

In [16]:
def evaluate_averaged_global_model(global_model, C_mat_test, B_vec_test, num_nodes):
    C_mat_test_partitioned = np.array_split(C_mat_test, num_nodes, axis=1)
    
    all_predictions = []
    
    for i in range(num_nodes):
        partition_preds = global_model.predict(C_mat_test_partitioned[i])
        all_predictions.append(partition_preds)
    
    avg_predictions = np.mean(all_predictions, axis=0)
    
    mse = mean_squared_error(B_vec_test, avg_predictions)
    
    return avg_predictions, mse

In [17]:
def train_global_model(num_nodes, C_train_partitioned):
    local_weights = []
    local_intercepts = []

    for i in range(num_nodes):
        node_model = Ridge(alpha=ALPHA)
        node_model.fit(C_train_partitioned[i], B_train)
        
        local_weights.append(node_model.coef_)
        local_intercepts.append(node_model.intercept_)

    global_weights_avg = np.mean(local_weights, axis=0)
    global_intercept_avg = np.mean(local_intercepts)

    global_model = Ridge()
    global_model.coef_ = global_weights_avg
    global_model.intercept_ = global_intercept_avg

    return global_model



In [24]:
def evaluate_fed_avg(num_nodes):
    C_train_partitioned = np.array_split(C_train, num_nodes, axis=1)
    C_test_partitioned = np.array_split(C_test, num_nodes, axis=1)
    global_model = train_global_model(num_nodes, C_train_partitioned)

    avg_predictions, final_mse = evaluate_averaged_global_model(
        global_model, 
        C_test, 
        B_test, 
        num_nodes
    )

    print(f"Final Averaged Global MSE: {final_mse:.4f}")

In [25]:
evaluate_fed_avg(10)

Final Averaged Global MSE: 3.3166


In [26]:
evaluate_fed_avg(5)

Final Averaged Global MSE: 3.4435


In [27]:
evaluate_fed_avg(3)

Final Averaged Global MSE: 2.8247


In [28]:
evaluate_fed_avg(1)

Final Averaged Global MSE: 0.5112
